# PyTorch Troubleshooting Simulation

This notebook represents a simple ML pipline, loading data, defining a model and fitting the model to the data. It has also been instrumented with Weights and Biases logging tools.

Your task is to debug this simple pipeline such that the model is able to learn and <u>perform reasonably well</u> on the given task, without changing the general structure of the model. As you do so, use comments and markdown cells to explain a bit about your process.


In [1]:
%pip install --quiet wandb



[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader

from torchvision import datasets, transforms

import wandb


## Prepare the image data


In [3]:
# This transform is intentionally declared separately from the dataset below.
configured_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5))
])

examples_per_batch = 32
all_images = datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=configured_transform,
)

split_index = 40_000
generator = torch.Generator().manual_seed(42)
training_images, validation_images = torch.utils.data.random_split(all_images, [40000, 10000], generator) # shuffles the sampleset 

training_loader = DataLoader(training_images, batch_size=examples_per_batch)
validation_loader = DataLoader(validation_images, batch_size=examples_per_batch)


In [4]:
class CifarClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.feature_one = nn.Conv2d(3, 6, 5)
        self.downsample = nn.MaxPool2d(2, 2)
        self.feature_two = nn.Conv2d(6, 16, 5)
        self.hidden_layer = nn.Linear(400, 120)
        self.narrow_layer = nn.Linear(120, 80)
        self.output_layer = nn.Linear(80, 10)

    def forward(self, batch):
        batch = self.downsample(F.relu(self.feature_one(batch)))
        batch = self.downsample(F.relu(self.feature_two(batch)))
        batch = torch.flatten(batch, 1)
        batch = F.relu(self.hidden_layer(batch))
        batch = F.relu(self.narrow_layer(batch))
        return self.output_layer(batch)


model = CifarClassifier()


In [5]:
loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


## Training and validation

Alongside loss, calculate both training and validation accuracy and send them to Weights & Biases.


In [ ]:
with wandb.init(project="PyTorch_Test", save_code=True) as run:
    for epoch_number in range(5):
        
        accumulated_loss_t = 0
        correct_t = 0 # Adding for accuracy calculations
        total_t = 0 # Adding for accuracy calculations
        model.train()

        for batch_number_t, (images, targets) in enumerate(training_loader):
            optimizer.zero_grad()
            
            predictions = model(images)
            batch_loss = loss_function(predictions, targets)

            batch_loss.backward()
            optimizer.step()

            accumulated_loss_t += batch_loss.item()

            _, predicted = torch.max(predictions, 1) # Calculating predictions
            total_t += targets.size(0) # count samples in batch across epoch
            correct_t += (predicted == targets).sum().item() # Tracking correct predictions

        model.eval()
        accumulated_loss_v = 0
        correct_v = 0 # Adding for accuracy calculations
        total_v = 0 # Adding for accuracy calculations    
        with torch.no_grad():
            for batch_number_v, (images, targets) in enumerate(validation_loader):
                predictions = model(images)

                batch_loss = loss_function(predictions, targets)
                
                accumulated_loss_v += batch_loss.item()

                _, predicted = torch.max(predictions, 1) # Calculating predictions
                total_v += targets.size(0) # count samples in batch across epoch
                correct_v += (predicted == targets).sum().item() # Tracking correct predictions

        run.log({
            'train_loss': accumulated_loss_t / (batch_number_t + 1),
            'train_acc': 100 * correct_t / total_t,
            'val_loss': accumulated_loss_v / (batch_number_v + 1),
            'val_acc': 100 * correct_v / total_v
        })


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/dbearsong/.netrc.
wandb: Currently logged in as: dkbearsong (dkbearsong-none) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


train_acc,▁▅▆▇█
train_loss,█▄▃▂▁
val_acc,▁▄▆▇█
val_loss,█▄▂▁▁
train_acc,62.4575
train_loss,1.05583
val_acc,60.42
val_loss,1.15657
